# Clustering Deret Waktu Transaksi Penjualan dengan DTW

Notebook ini menjalankan eksperimen DTW, CDTW, DDTW, dan WDTW pada tiga representasi data, kemudian membandingkan K-Means DBA, K-Medoids, Hierarchical, dan DBSCAN.

**Catatan:** K-Means DBA menggunakan `TimeSeriesKMeans` dari `tslearn`. Silhouette dihitung dari matriks jarak asli, sedangkan CHI dan DBI dihitung pada koordinat MDS 5D.

Jalankan cell secara berurutan. Sesuaikan lokasi file Excel pada Cell 2.

## CELL 1. INSTALL (Google Colab)

Jalankan instalasi jika package belum tersedia.

In [1]:
!pip install -q tslearn scikit-learn-extra numba openpyxl


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Jalankan jika package belum tersedia:

import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from numba import njit, prange
from tslearn.metrics import cdist_dtw
from tslearn.clustering import TimeSeriesKMeans

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import MDS
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score,
    davies_bouldin_score, adjusted_rand_score
)
from sklearn_extra.cluster import KMedoids

from scipy.cluster.hierarchy import linkage, cut_tree
from scipy.spatial.distance import squareform

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
OUTPUT_DIR = "hasil_clustering_dtw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

from google.colab import files
import warnings
warnings.filterwarnings('ignore')

from google.colab import files
from io import BytesIO

# Pengaturan tampilan
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("Library berhasil diimport.")

Library berhasil diimport.


## CELL 2. LOAD DATA

In [3]:
# Upload dataset
uploaded = files.upload()

# Membaca file Excel
file_name = list(uploaded.keys())[0]
df = pd.read_excel(BytesIO(uploaded[file_name]))

# Menampilkan beberapa baris pertama
df.head()

Saving Sales transaction data.xlsx to Sales transaction data.xlsx


,Product_Code,W0,W1,W2,W3,W4,W5,W6,W7,W8,W9,W10,W11,W12,W13,W14,W15,W16,W17,W18,W19,W20,W21,W22,W23,W24,W25,W26,W27,W28,W29,W30,W31,W32,W33,W34,W35,W36,W37,W38,W39,W40,W41,W42,W43,W44,W45,W46,W47,W48,W49,W50,W51,MIN,MAX,Normalized 0,Normalized 1,Normalized 2,Normalized 3,Normalized 4,Normalized 5,Normalized 6,Normalized 7,Normalized 8,Normalized 9,Normalized 10,Normalized 11,Normalized 12,Normalized 13,Normalized 14,Normalized 15,Normalized 16,Normalized 17,Normalized 18,Normalized 19,Normalized 20,Normalized 21,Normalized 22,Normalized 23,Normalized 24,Normalized 25,Normalized 26,Normalized 27,Normalized 28,Normalized 29,Normalized 30,Normalized 31,Normalized 32,Normalized 33,Normalized 34,Normalized 35,Normalized 36,Normalized 37,Normalized 38,Normalized 39,Normalized 40,Normalized 41,Normalized 42,Normalized 43,Normalized 44,Normalized 45,Normalized 46,Normalized 47,Normalized 48,Normalized 49,Normalized 50,Normalized 51
0,P1,11,12,10,8,13,12,14,21,6,14,11,14,16,9,9,9,14,9,3,12,5,11,7,12,5,9,7,10,5,11,7,10,12,6,5,14,10,9,12,17,7,11,4,7,8,10,12,3,7,6,5,10,3,21,0.44,0.50,0.39,0.28,0.56,0.50,0.61,1.00,0.17,0.61,0.44,0.61,0.72,0.33,0.33,0.33,0.61,0.33,0.00,0.50,0.11,0.44,0.22,0.50,0.11,0.33,0.22,0.39,0.11,0.44,0.22,0.39,0.50,0.17,0.11,0.61,0.39,0.33,0.50,0.78,0.22,0.44,0.06,0.22,0.28,0.39,0.50,0.00,0.22,0.17,0.11,0.39
1,P2,7,6,3,2,7,1,6,3,3,3,2,2,6,2,0,6,2,7,7,9,4,7,2,4,5,3,5,8,5,5,3,1,3,2,3,10,5,2,7,3,2,5,2,4,5,1,1,4,5,1,6,0,0,10,0.70,0.60,0.30,0.20,0.70,0.10,0.60,0.30,0.30,0.30,0.20,0.20,0.60,0.20,0.00,0.60,0.20,0.70,0.70,0.90,0.40,0.70,0.20,0.40,0.50,0.30,0.50,0.80,0.50,0.50,0.30,0.10,0.30,0.20,0.30,1.00,0.50,0.20,0.70,0.30,0.20,0.50,0.20,0.40,0.50,0.10,0.10,0.40,0.50,0.10,0.60,0.00
2,P3,7,11,8,9,10,8,7,13,12,6,14,9,4,7,12,8,7,11,10,7,7,13,11,8,10,8,14,5,3,13,11,9,7,8,7,9,6,12,12,9,3,5,6,14,5,5,7,8,14,8,8,7,3,14,0.36,0.73,0.45,0.55,0.64,0.45,0.36,0.91,0.82,0.27,1.00,0.55,0.09,0.36,0.82,0.45,0.36,0.73,0.64,0.36,0.36,0.91,0.73,0.45,0.64,0.45,1.00,0.18,0.00,0.91,0.73,0.55,0.36,0.45,0.36,0.55,0.27,0.82,0.82,0.55,0.00,0.18,0.27,1.00,0.18,0.18,0.36,0.45,1.00,0.45,0.45,0.36
3,P4,12,8,13,5,9,6,9,13,13,11,8,4,5,4,15,7,11,9,15,4,6,7,11,7,9,6,10,10,2,6,7,2,5,12,5,19,8,6,8,8,12,6,9,10,3,4,6,8,14,8,7,8,2,19,0.59,0.35,0.65,0.18,0.41,0.24,0.41,0.65,0.65,0.53,0.35,0.12,0.18,0.12,0.76,0.29,0.53,0.41,0.76,0.12,0.24,0.29,0.53,0.29,0.41,0.24,0.47,0.47,0.00,0.24,0.29,0.00,0.18,0.59,0.18,1.00,0.35,0.24,0.35,0.35,0.59,0.24,0.41,0.47,0.06,0.12,0.24,0.35,0.71,0.35,0.29,0.35
4,P5,8,5,13,11,6,7,9,14,9,9,11,18,8,4,13,8,10,15,6,13,11,6,10,9,8,12,8,9,13,3,5,3,5,5,9,7,4,8,8,5,5,8,7,11,7,12,6,6,5,11,8,9,3,18,0.33,0.13,0.67,0.53,0.20,0.27,0.40,0.73,0.40,0.40,0.53,1.00,0.33,0.07,0.67,0.33,0.47,0.80,0.20,0.67,0.53,0.20,0.47,0.40,0.33,0.60,0.33,0.40,0.67,0.00,0.13,0.00,0.13,0.13,0.40,0.27,0.07,0.33,0.33,0.13,0.13,0.33,0.27,0.53,0.27,0.60,0.20,0.20,0.13,0.53,0.33,0.40


In [4]:
# Jika di Colab, upload file lalu ganti path sesuai nama file.
FILE_PATH = "Sales transaction data.xlsx"
df = pd.read_excel(FILE_PATH)

weekly_cols = [f"W{i}" for i in range(52)]
normalized_cols = [f"Normalized {i}" for i in range(52)]

required_cols = ["Product_Code"] + weekly_cols + normalized_cols
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom berikut tidak ditemukan: {missing_cols}")

# Pastikan satu baris = satu produk dan data mingguan numerik.
if df["Product_Code"].duplicated().any():
    raise ValueError("Product_Code duplikat ditemukan. Periksa sebelum clustering.")

X_original = df[weekly_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
X_minmax = df[normalized_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)

if np.isnan(X_original).any() or np.isnan(X_minmax).any():
    raise ValueError("Terdapat missing value pada W0-W51 atau kolom Normalized.")

# Standardisasi per minggu/kolom, konsisten dengan StandardScaler
# pada matriks (produk x minggu).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_original)

product_code = df["Product_Code"].astype(str).to_numpy()

X_by_repr = {
    "Original": X_original,
    "MinMax": X_minmax,
    "Scaled": X_scaled
}

print("Ukuran data:", X_original.shape)
print("Jumlah produk:", len(product_code))


Ukuran data: (811, 52)
Jumlah produk: 811


## CELL 3. DDTW TRANSFORM

In [5]:
def derivative_ts(X):
    """Turunan diskrit untuk setiap produk."""
    X = np.asarray(X, dtype=float)
    out = np.zeros_like(X, dtype=float)
    out[:, 0] = X[:, 1] - X[:, 0]
    for t in range(1, X.shape[1] - 1):
        out[:, t] = (
            0.5 * (X[:, t] - X[:, t-1])
            + 0.25 * (X[:, t+1] - X[:, t-1])
        )
    out[:, -1] = X[:, -1] - X[:, -2]
    return out

X_derivative = {name: derivative_ts(X) for name, X in X_by_repr.items()}


## CELL 4. WDTW IMPLEMENTATION

In [6]:
def wdtw_weights(length, g):
    idx = np.arange(length)
    delta = np.abs(idx[:, None] - idx[None, :])
    return 1.0 / (1.0 + np.exp(-g * (delta - length / 2.0)))

@njit
def _wdtw_pair(x, y, weights):
    n = len(x)
    m = len(y)
    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = weights[i-1, j-1] * (x[i-1] - y[j-1])**2
            best_prev = min(dp[i-1, j], dp[i, j-1], dp[i-1, j-1])
            dp[i, j] = cost + best_prev
    return np.sqrt(dp[n, m])

@njit(parallel=True)
def _wdtw_matrix(X, weights):
    n = X.shape[0]
    D = np.zeros((n, n))
    for i in prange(n):
        for j in range(i + 1, n):
            val = _wdtw_pair(X[i], X[j], weights)
            D[i, j] = val
            D[j, i] = val
    return D


## CELL 5. COMPUTE / SAVE ALL DISTANCE MATRICES

In [7]:
# Total: 3 DTW + 9 CDTW + 3 DDTW + 12 WDTW = 27 matrices.
# Matriks disimpan sekali dan dipakai ulang oleh semua algoritma.

distance_matrices = {}
distance_times = {}

def add_distance(name, D, elapsed):
    D = np.asarray(D, dtype=np.float64)
    D = (D + D.T) / 2.0
    np.fill_diagonal(D, 0.0)
    if D.shape != (len(product_code), len(product_code)):
        raise ValueError(f"Ukuran matriks {name} tidak sesuai: {D.shape}")
    if not np.isfinite(D).all() or (D < -1e-9).any():
        raise ValueError(f"Matriks {name} mengandung nilai tidak valid.")
    distance_matrices[name] = D
    distance_times[name] = elapsed

# DTW standard
for rep, X in X_by_repr.items():
    name = f"DTW_{rep}"
    t0 = time.perf_counter()
    D = cdist_dtw(X)
    add_distance(name, D, time.perf_counter() - t0)
    print(name, D.shape, round(distance_times[name], 2), "detik")

# Constrained DTW (Sakoe-Chiba radius)
for rep, X in X_by_repr.items():
    for radius in [2, 3, 5]:
        name = f"CDTW_{rep}_R{radius}"
        t0 = time.perf_counter()
        D = cdist_dtw(
            X,
            global_constraint="sakoe_chiba",
            sakoe_chiba_radius=radius
        )
        add_distance(name, D, time.perf_counter() - t0)
        print(name, D.shape, round(distance_times[name], 2), "detik")

# DDTW: DTW pada deret turunan
for rep, Xd in X_derivative.items():
    name = f"DDTW_{rep}"
    t0 = time.perf_counter()
    D = cdist_dtw(Xd)
    add_distance(name, D, time.perf_counter() - t0)
    print(name, D.shape, round(distance_times[name], 2), "detik")

# WDTW
for rep, X in X_by_repr.items():
    for g in [0.05, 0.1, 0.5, 1.0]:
        name = f"WDTW_{rep}_g{g}"
        t0 = time.perf_counter()
        W = wdtw_weights(X.shape[1], g)
        D = _wdtw_matrix(np.ascontiguousarray(X, dtype=np.float64), W)
        add_distance(name, D, time.perf_counter() - t0)
        print(name, D.shape, round(distance_times[name], 2), "detik")

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "all_distance_matrices.npz"),
    **distance_matrices,
    product_code=product_code
)
pd.DataFrame(
    [{"Matriks": k, "Waktu_Detik": v} for k, v in distance_times.items()]
).to_csv(os.path.join(OUTPUT_DIR, "waktu_perhitungan_jarak.csv"), index=False)

print("Jumlah matriks jarak:", len(distance_matrices))


DTW_Original (811, 811) 20.12 detik
DTW_MinMax (811, 811) 11.39 detik
DTW_Scaled (811, 811) 11.18 detik
CDTW_Original_R2 (811, 811) 9.41 detik
CDTW_Original_R3 (811, 811) 6.31 detik
CDTW_Original_R5 (811, 811) 7.71 detik
CDTW_MinMax_R2 (811, 811) 5.97 detik
CDTW_MinMax_R3 (811, 811) 7.65 detik
CDTW_MinMax_R5 (811, 811) 6.53 detik
CDTW_Scaled_R2 (811, 811) 7.32 detik
CDTW_Scaled_R3 (811, 811) 6.36 detik
CDTW_Scaled_R5 (811, 811) 7.76 detik
DDTW_Original (811, 811) 11.25 detik
DDTW_MinMax (811, 811) 10.51 detik
DDTW_Scaled (811, 811) 11.15 detik
WDTW_Original_g0.05 (811, 811) 5.39 detik
WDTW_Original_g0.1 (811, 811) 3.11 detik
WDTW_Original_g0.5 (811, 811) 3.25 detik
WDTW_Original_g1.0 (811, 811) 3.03 detik
WDTW_MinMax_g0.05 (811, 811) 3.76 detik
WDTW_MinMax_g0.1 (811, 811) 3.08 detik
WDTW_MinMax_g0.5 (811, 811) 3.18 detik
WDTW_MinMax_g1.0 (811, 811) 3.02 detik
WDTW_Scaled_g0.05 (811, 811) 3.53 detik
WDTW_Scaled_g0.1 (811, 811) 3.3 detik
WDTW_Scaled_g0.5 (811, 811) 3.18 detik
WDTW_Scaled

## CELL 6. LOAD SAVED MATRICES (jalankan ini jika matriks sudah pernah dihitung)

In [8]:
# saved = np.load(os.path.join(OUTPUT_DIR, "all_distance_matrices.npz"))
# distance_matrices = {k: saved[k] for k in saved.files if k != "product_code"}
# product_code = saved["product_code"]


## CELL 7. MDS EMBEDDING (sekali per matriks, digunakan ulang)

In [9]:
# MDS bisa memakan waktu cukup besar. Jangan hitung ulang pada setiap k.
MDS_DIM = 5
mds_coordinates = {}
mds_stress = {}

for name, D in distance_matrices.items():
    print("MDS:", name)
    mds = MDS(
        n_components=MDS_DIM,
        metric=True,
        dissimilarity="precomputed",
        random_state=RANDOM_STATE,
        n_init=1,
        max_iter=200
    )
    X_mds = mds.fit_transform(D)
    mds_coordinates[name] = X_mds
    mds_stress[name] = getattr(mds, "stress_", np.nan)


MDS: DTW_Original
MDS: DTW_MinMax
MDS: DTW_Scaled
MDS: CDTW_Original_R2
MDS: CDTW_Original_R3
MDS: CDTW_Original_R5
MDS: CDTW_MinMax_R2
MDS: CDTW_MinMax_R3
MDS: CDTW_MinMax_R5
MDS: CDTW_Scaled_R2
MDS: CDTW_Scaled_R3
MDS: CDTW_Scaled_R5
MDS: DDTW_Original
MDS: DDTW_MinMax
MDS: DDTW_Scaled
MDS: WDTW_Original_g0.05
MDS: WDTW_Original_g0.1
MDS: WDTW_Original_g0.5
MDS: WDTW_Original_g1.0
MDS: WDTW_MinMax_g0.05
MDS: WDTW_MinMax_g0.1
MDS: WDTW_MinMax_g0.5
MDS: WDTW_MinMax_g1.0
MDS: WDTW_Scaled_g0.05
MDS: WDTW_Scaled_g0.1
MDS: WDTW_Scaled_g0.5
MDS: WDTW_Scaled_g1.0


## CELL 8. EVALUATION HELPERS

In [10]:
def cluster_distribution(labels, include_noise=False):
    labels = np.asarray(labels)
    if include_noise:
        vals = labels
    else:
        vals = labels[labels != -1]
    if len(vals) == 0:
        return {}, np.nan, np.nan, np.nan
    counts = pd.Series(vals).value_counts().sort_index()
    props = counts / counts.sum()
    if len(counts) <= 1:
        balance = 0.0
    else:
        entropy = -(props * np.log(props)).sum()
        balance = entropy / np.log(len(counts))
    return counts.to_dict(), int(counts.min()), int(counts.max()), float(balance)

def evaluate_labels(D, X_mds, labels, ignore_noise=False):
    labels = np.asarray(labels)
    mask = labels != -1 if ignore_noise else np.ones(len(labels), dtype=bool)
    y = labels[mask]
    n_clusters = len(np.unique(y))
    n_obs = len(y)

    out = {
        "Silhouette": np.nan,
        "CHI_MDS": np.nan,
        "DBI_MDS": np.nan,
        "Jumlah_Cluster": n_clusters,
        "Jumlah_Dievaluasi": n_obs
    }

    # Metrik valid jika terdapat 2 sampai n-1 cluster
    if n_clusters >= 2 and n_clusters < n_obs:
        D_sub = D[np.ix_(mask, mask)]
        X_sub = X_mds[mask]
        out["Silhouette"] = silhouette_score(
            D_sub, y, metric="precomputed"
        )
        out["CHI_MDS"] = calinski_harabasz_score(X_sub, y)
        out["DBI_MDS"] = davies_bouldin_score(X_sub, y)

    dist, cmin, cmax, balance = cluster_distribution(
        labels, include_noise=not ignore_noise
    )
    out.update({
        "Cluster_Terkecil": cmin,
        "Cluster_Terbesar": cmax,
        "Balance_Score": balance,
        "Distribusi": str(dist)
    })
    if ignore_noise:
        out["Persentase_Noise"] = float((labels == -1).mean() * 100)
    else:
        out["Persentase_Noise"] = 0.0
    return out


## CELL 9. K-MEANS WITH DBA (REAL TIME-SERIES K-MEANS)

In [11]:
# Dibandingkan dengan KMeans+MDS pada notebook lama:
# ini benar-benar memperbarui centroid menggunakan DBA.
#
# KMeans DBA tersedia untuk:
# - DTW standar
# - CDTW dengan Sakoe-Chiba
# - DDTW pada data turunan (DBA centroid di ruang turunan)
#
# WDTW tidak dijalankan di bagian ini karena tslearn tidak menyediakan
# WDTW barycenter/DBA bawaan. Jangan menggantinya diam-diam dengan KMeans+MDS.

k_values = range(2, 11)
kmeans_rows = []
kmeans_labels = {}

for rep, X in X_by_repr.items():
    X3 = X[:, :, None]
    for variant in ["DTW", "CDTW_R2", "CDTW_R3", "CDTW_R5", "DDTW"]:
        if variant == "DTW":
            X_input = X3
            metric_params = {}
            dist_name = f"DTW_{rep}"
        elif variant.startswith("CDTW"):
            radius = int(variant.split("R")[1])
            X_input = X3
            metric_params = {
                "global_constraint": "sakoe_chiba",
                "sakoe_chiba_radius": radius
            }
            dist_name = f"CDTW_{rep}_R{radius}"
        else:
            X_input = X_derivative[rep][:, :, None]
            metric_params = {}
            dist_name = f"DDTW_{rep}"

        D = distance_matrices[dist_name]
        X_mds = mds_coordinates[dist_name]

        for k in k_values:
            t0 = time.perf_counter()
            model = TimeSeriesKMeans(
                n_clusters=k,
                metric="dtw",
                metric_params=metric_params,
                max_iter=10,
                max_iter_barycenter=10,
                n_init=2,
                n_jobs=-1,
                random_state=RANDOM_STATE,
                verbose=False
            )
            labels = model.fit_predict(X_input)
            elapsed = time.perf_counter() - t0

            metrics = evaluate_labels(D, X_mds, labels)
            row = {
                "Algoritma": "KMeans_DBA",
                "Representasi": rep,
                "Varian": variant,
                "Matriks": dist_name,
                "Parameter": "-",
                "K": k,
                "Running_Time": elapsed,
                **metrics
            }
            kmeans_rows.append(row)
            kmeans_labels[(dist_name, k)] = labels
            print(dist_name, "k=", k, "sil=", round(metrics["Silhouette"], 4))

tabel_kmeans_dba = pd.DataFrame(kmeans_rows)


DTW_Original k= 2 sil= 0.7804
DTW_Original k= 3 sil= 0.6852
DTW_Original k= 4 sil= 0.6071
DTW_Original k= 5 sil= 0.5037
DTW_Original k= 6 sil= 0.4875
DTW_Original k= 7 sil= 0.3845
DTW_Original k= 8 sil= 0.4052
DTW_Original k= 9 sil= 0.3369
DTW_Original k= 10 sil= 0.33
CDTW_Original_R2 k= 2 sil= 0.7633
CDTW_Original_R2 k= 3 sil= 0.6739
CDTW_Original_R2 k= 4 sil= 0.6284
CDTW_Original_R2 k= 5 sil= 0.5531
CDTW_Original_R2 k= 6 sil= 0.4688
CDTW_Original_R2 k= 7 sil= 0.4146
CDTW_Original_R2 k= 8 sil= 0.4171
CDTW_Original_R2 k= 9 sil= 0.4101
CDTW_Original_R2 k= 10 sil= 0.276
CDTW_Original_R3 k= 2 sil= 0.7679
CDTW_Original_R3 k= 3 sil= 0.6739
CDTW_Original_R3 k= 4 sil= 0.6297
CDTW_Original_R3 k= 5 sil= 0.5611
CDTW_Original_R3 k= 6 sil= 0.4733
CDTW_Original_R3 k= 7 sil= 0.4196
CDTW_Original_R3 k= 8 sil= 0.4535
CDTW_Original_R3 k= 9 sil= 0.4194
CDTW_Original_R3 k= 10 sil= 0.3847
CDTW_Original_R5 k= 2 sil= 0.772
CDTW_Original_R5 k= 3 sil= 0.6874
CDTW_Original_R5 k= 4 sil= 0.6249
CDTW_Original_R5 

## CELL 10. K-MEDOIDS PAM (PRECOMPUTED DTW)

In [12]:
kmedoids_rows = []
kmedoids_labels = {}

for name, D in distance_matrices.items():
    X_mds = mds_coordinates[name]
    for k in k_values:
        t0 = time.perf_counter()
        model = KMedoids(
            n_clusters=k,
            metric="precomputed",
            method="pam",
            init="k-medoids++",
            random_state=RANDOM_STATE
        )
        labels = model.fit_predict(D)
        elapsed = time.perf_counter() - t0

        metrics = evaluate_labels(D, X_mds, labels)
        kmedoids_rows.append({
            "Algoritma": "KMedoids_PAM",
            "Representasi": name.split("_")[1],
            "Varian": name,
            "Matriks": name,
            "Parameter": "-",
            "K": k,
            "Running_Time": elapsed,
            **metrics
        })
        kmedoids_labels[(name, k)] = labels

tabel_kmedoids = pd.DataFrame(kmedoids_rows)


In [22]:
tabel_kmedoids

,Algoritma,Representasi,Varian,Matriks,Parameter,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise
0,KMedoids_PAM,Original,DTW_Original,DTW_Original,-,2,0.08,0.78,3596.27,0.37,2,811,158,653,0.71,"{0: 653, 1: 158}",0.00
1,KMedoids_PAM,Original,DTW_Original,DTW_Original,-,3,0.10,0.68,6251.63,0.43,3,811,126,471,0.87,"{0: 214, 1: 126, 2: 471}",0.00
2,KMedoids_PAM,Original,DTW_Original,DTW_Original,-,4,0.16,0.59,6528.69,0.43,4,811,48,464,0.80,"{0: 176, 1: 123, 2: 464, 3: 48}",0.00
3,KMedoids_PAM,Original,DTW_Original,DTW_Original,-,5,0.23,0.52,7774.85,0.42,5,811,47,299,0.91,"{0: 151, 1: 123, 2: 299, 3: 47, 4: 191}",0.00
4,KMedoids_PAM,Original,DTW_Original,DTW_Original,-,6,0.31,0.48,7064.56,0.53,6,811,40,299,0.88,"{0: 115, 1: 121, 2: 299, 3: 40, 4: 46, 5: 190}",0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,KMedoids_PAM,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,-,6,0.32,0.45,5.63,13.69,6,811,45,302,0.89,"{0: 97, 1: 120, 2: 188, 3: 45, 4: 59, 5: 302}",0.00
239,KMedoids_PAM,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,-,7,0.36,0.38,5.70,13.23,7,811,42,302,0.87,"{0: 42, 1: 45, 2: 188, 3: 78, 4: 59, 5: 302, 6...",0.00
240,KMedoids_PAM,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,-,8,0.50,0.27,4.89,12.88,8,811,42,249,0.91,"{0: 97, 1: 45, 2: 249, 3: 79, 4: 162, 5: 59, 6...",0.00
241,KMedoids_PAM,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,-,9,0.53,0.26,4.27,13.43,9,811,19,249,0.88,"{0: 97, 1: 43, 2: 249, 3: 25, 4: 162, 5: 59, 6...",0.00


## CELL 11. HIERARCHICAL (AVERAGE & COMPLETE)

In [23]:
hier_rows = []
hier_labels = {}
linkage_methods = ["average", "complete"]

for name, D in distance_matrices.items():
    X_mds = mds_coordinates[name]
    condensed = squareform(D, checks=False)

    for link_method in linkage_methods:
        t0 = time.perf_counter()
        Z = linkage(condensed, method=link_method)
        fit_time = time.perf_counter() - t0

        for k in k_values:
            labels = cut_tree(Z, n_clusters=[k]).ravel()
            metrics = evaluate_labels(D, X_mds, labels)
            hier_rows.append({
                "Algoritma": "Hierarchical",
                "Representasi": name.split("_")[1],
                "Varian": name,
                "Matriks": name,
                "Parameter": link_method,
                "Linkage": link_method,
                "K": k,
                "Running_Time": fit_time,
                **metrics
            })
            hier_labels[(name, link_method, k)] = labels

tabel_hierarchical = pd.DataFrame(hier_rows)

KeyboardInterrupt: 

In [24]:
tabel_hierarchical

,Algoritma,Representasi,Varian,Matriks,Parameter,Linkage,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise
0,Hierarchical,Original,DTW_Original,DTW_Original,average,average,2,0.02,0.77,3144.74,0.27,2,811,120,691,0.60,"{0: 691, 1: 120}",0.00
1,Hierarchical,Original,DTW_Original,DTW_Original,average,average,3,0.02,0.73,1581.15,0.29,3,811,1,691,0.39,"{0: 691, 1: 119, 2: 1}",0.00
2,Hierarchical,Original,DTW_Original,DTW_Original,average,average,4,0.02,0.66,4175.99,0.40,4,811,1,490,0.68,"{0: 201, 1: 490, 2: 119, 3: 1}",0.00
3,Hierarchical,Original,DTW_Original,DTW_Original,average,average,5,0.02,0.62,3151.13,0.80,5,811,1,490,0.59,"{0: 201, 1: 490, 2: 117, 3: 2, 4: 1}",0.00
4,Hierarchical,Original,DTW_Original,DTW_Original,average,average,6,0.02,0.57,4193.96,0.72,6,811,1,490,0.61,"{0: 156, 1: 490, 2: 45, 3: 117, 4: 2, 5: 1}",0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,Hierarchical,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,complete,complete,6,0.02,0.54,9.77,7.31,6,811,1,467,0.62,"{0: 179, 1: 467, 2: 45, 3: 117, 4: 2, 5: 1}",0.00
482,Hierarchical,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,complete,complete,7,0.02,0.53,8.79,6.62,7,811,1,467,0.58,"{0: 179, 1: 467, 2: 40, 3: 117, 4: 2, 5: 1, 6: 5}",0.00
483,Hierarchical,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,complete,complete,8,0.02,0.50,9.66,6.37,8,811,1,467,0.59,"{0: 179, 1: 467, 2: 40, 3: 36, 4: 81, 5: 2, 6:...",0.00
484,Hierarchical,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,complete,complete,9,0.02,0.50,9.60,6.22,9,811,1,467,0.57,"{0: 179, 1: 467, 2: 40, 3: 26, 4: 81, 5: 10, 6...",0.00


## CELL 12. DBSCAN (PRECOMPUTED)

In [14]:
# Epsilon dibuat spesifik untuk setiap matriks dari k-distance.
dbscan_rows = []
dbscan_labels = {}
min_samples_values = [5, 10, 15]
eps_percentiles = [50, 60, 70, 80, 90, 95]

for name, D in distance_matrices.items():
    X_mds = mds_coordinates[name]

    for min_samples in min_samples_values:
        # min_samples-th nearest-neighbor distance, tanpa menghitung diri sendiri
        sorted_D = np.sort(D, axis=1)
        kth_dist = sorted_D[:, min_samples]  # kolom 0 adalah jarak ke diri sendiri
        eps_values = np.unique(np.percentile(kth_dist, eps_percentiles))

        for eps in eps_values:
            t0 = time.perf_counter()
            model = DBSCAN(
                eps=float(eps),
                min_samples=min_samples,
                metric="precomputed"
            )
            labels = model.fit_predict(D)
            elapsed = time.perf_counter() - t0

            metrics = evaluate_labels(
                D, X_mds, labels, ignore_noise=True
            )
            dbscan_rows.append({
                "Algoritma": "DBSCAN",
                "Representasi": name.split("_")[1],
                "Varian": name,
                "Matriks": name,
                "Parameter": f"eps={eps:.6g}; min_samples={min_samples}",
                "Epsilon": float(eps),
                "Min_Samples": min_samples,
                "Running_Time": elapsed,
                **metrics
            })
            dbscan_labels[(name, float(eps), min_samples)] = labels

tabel_dbscan = pd.DataFrame(dbscan_rows)


In [25]:
tabel_dbscan

,Algoritma,Representasi,Varian,Matriks,Parameter,Epsilon,Min_Samples,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise
0,DBSCAN,Original,DTW_Original,DTW_Original,eps=9.64365; min_samples=5,9.64,5,0.04,NaN,NaN,NaN,1,436,436,436,0.00,{0: 436},46.24
1,DBSCAN,Original,DTW_Original,DTW_Original,eps=13.0767; min_samples=5,13.08,5,0.02,0.61,101.14,0.35,2,498,8,490,0.12,"{0: 490, 1: 8}",38.59
2,DBSCAN,Original,DTW_Original,DTW_Original,eps=16.0624; min_samples=5,16.06,5,0.02,NaN,NaN,NaN,1,594,594,594,0.00,{0: 594},26.76
3,DBSCAN,Original,DTW_Original,DTW_Original,eps=22.0454; min_samples=5,22.05,5,0.02,NaN,NaN,NaN,1,664,664,664,0.00,{0: 664},18.13
4,DBSCAN,Original,DTW_Original,DTW_Original,eps=30.6268; min_samples=5,30.63,5,0.02,0.77,2287.63,0.26,2,770,82,688,0.49,"{0: 688, 1: 82}",5.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,DBSCAN,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,eps=4.36221e-06; min_samples=15,0.00,15,0.01,0.62,6.47,5.29,2,511,20,491,0.24,"{0: 491, 1: 20}",36.99
482,DBSCAN,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,eps=5.20991e-06; min_samples=15,0.00,15,0.01,NaN,NaN,NaN,1,615,615,615,0.00,{0: 615},24.17
483,DBSCAN,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,eps=7.0453e-06; min_samples=15,0.00,15,0.01,NaN,NaN,NaN,1,674,674,674,0.00,{0: 674},16.89
484,DBSCAN,Scaled,WDTW_Scaled_g1.0,WDTW_Scaled_g1.0,eps=9.79574e-06; min_samples=15,0.00,15,0.01,0.76,29.00,4.99,2,790,102,688,0.56,"{0: 688, 1: 102}",2.59


## CELL 13. COMBINE RESULTS

In [15]:
tables = [
    tabel_kmeans_dba,
    tabel_kmedoids,
    tabel_hierarchical,
    tabel_dbscan
]
tabel_semua = pd.concat(tables, ignore_index=True, sort=False)

# Validasi: hanya hasil yang mempunyai tiga metrik valid
valid = tabel_semua.dropna(
    subset=["Silhouette", "CHI_MDS", "DBI_MDS"]
).copy()

# Ranking global. Karena CHI/DBI berasal dari MDS, interpretasikan
# sebagai metrik pendukung; utamakan Silhouette dari matriks jarak asli.
valid["Rank_SI"] = valid["Silhouette"].rank(ascending=False, method="average")
valid["Rank_CHI"] = valid["CHI_MDS"].rank(ascending=False, method="average")
valid["Rank_DBI"] = valid["DBI_MDS"].rank(ascending=True, method="average")

# Rata-rata ranking tanpa bobot (lebih transparan sebagai baseline).
valid["Skor_Ranking"] = valid[
    ["Rank_SI", "Rank_CHI", "Rank_DBI"]
].mean(axis=1)
valid["Rank_Akhir"] = valid["Skor_Ranking"].rank(
    ascending=True, method="min"
)

# Simpan semua tabel
tabel_semua.to_csv(os.path.join(OUTPUT_DIR, "evaluasi_seluruh_model.csv"), index=False)
valid.sort_values("Rank_Akhir").to_csv(
    os.path.join(OUTPUT_DIR, "ranking_model_valid.csv"), index=False
)

print("\nTOP 3 KOMBINASI KESELURUHAN")
display(valid.sort_values("Rank_Akhir").head(3))



TOP 3 KOMBINASI KESELURUHAN


,Algoritma,Representasi,Varian,Matriks,Parameter,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise,Linkage,Epsilon,Min_Samples,Rank_SI,Rank_CHI,Rank_DBI,Skor_Ranking,Rank_Akhir
468,Hierarchical,Original,CDTW_Original_R5,CDTW_Original_R5,average,2.00,0.01,0.78,3469.48,0.28,2,811,124,687,0.62,"{0: 687, 1: 124}",0.00,average,NaN,NaN,7.00,205.00,62.00,91.33,1.00
971,DBSCAN,Original,CDTW_Original_R5,CDTW_Original_R5,eps=35.4894; min_samples=15,NaN,0.01,0.78,3246.21,0.27,2,805,116,689,0.59,"{0: 689, 1: 116}",0.74,NaN,35.49,15.00,9.00,246.00,28.00,94.33,2.00
953,DBSCAN,Original,CDTW_Original_R3,CDTW_Original_R3,eps=37.0134; min_samples=15,NaN,0.02,0.78,3329.58,0.28,2,803,116,687,0.60,"{0: 687, 1: 116}",0.99,NaN,37.01,15.00,3.00,233.00,52.00,96.00,3.00


## CELL 14. PERBANDINGAN PER REPRESENTASI

In [16]:
# Representasi diekstrak dari nama matriks, agar semua 4 algoritma
# bisa diringkas pada Original / MinMax / Scaled.
def get_representation(matrix_name):
    if "_Original" in matrix_name:
        return "Original"
    if "_MinMax" in matrix_name:
        return "MinMax"
    if "_Scaled" in matrix_name:
        return "Scaled"
    return "Unknown"

valid["Kelompok_Data"] = valid["Matriks"].apply(get_representation)

for rep in ["Original", "MinMax", "Scaled"]:
    print(f"\nTOP 10 - DATA {rep.upper()}")
    display(
        valid[valid["Kelompok_Data"] == rep]
        .sort_values("Rank_Akhir")
        .head(10)
    )



TOP 10 - DATA ORIGINAL


,Algoritma,Representasi,Varian,Matriks,Parameter,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise,Linkage,Epsilon,Min_Samples,Rank_SI,Rank_CHI,Rank_DBI,Skor_Ranking,Rank_Akhir,Kelompok_Data
468,Hierarchical,Original,CDTW_Original_R5,CDTW_Original_R5,average,2.00,0.01,0.78,3469.48,0.28,2,811,124,687,0.62,"{0: 687, 1: 124}",0.00,average,NaN,NaN,7.00,205.00,62.00,91.33,1.00,Original
971,DBSCAN,Original,CDTW_Original_R5,CDTW_Original_R5,eps=35.4894; min_samples=15,NaN,0.01,0.78,3246.21,0.27,2,805,116,689,0.59,"{0: 689, 1: 116}",0.74,NaN,35.49,15.00,9.00,246.00,28.00,94.33,2.00,Original
953,DBSCAN,Original,CDTW_Original_R3,CDTW_Original_R3,eps=37.0134; min_samples=15,NaN,0.02,0.78,3329.58,0.28,2,803,116,687,0.60,"{0: 687, 1: 116}",0.99,NaN,37.01,15.00,3.00,233.00,52.00,96.00,3.00,Original
648,Hierarchical,Original,WDTW_Original_g0.05,WDTW_Original_g0.05,average,2.00,0.01,0.78,3488.22,0.28,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,average,NaN,NaN,22.00,202.00,65.00,96.33,4.00,Original
965,DBSCAN,Original,CDTW_Original_R5,CDTW_Original_R5,eps=34.4093; min_samples=10,NaN,0.02,0.78,3216.15,0.27,2,804,115,689,0.59,"{0: 689, 1: 115}",0.86,NaN,34.41,10.00,10.00,253.00,27.00,96.67,5.00,Original
947,DBSCAN,Original,CDTW_Original_R3,CDTW_Original_R3,eps=36.318; min_samples=10,NaN,0.01,0.78,3309.58,0.28,2,802,115,687,0.59,"{0: 687, 1: 115}",1.11,NaN,36.32,10.00,1.00,240.00,51.00,97.33,6.00,Original
387,Hierarchical,Original,DTW_Original,DTW_Original,complete,2.00,0.01,0.78,3451.77,0.27,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,36.00,209.00,47.00,97.33,6.00,Original
675,Hierarchical,Original,WDTW_Original_g0.1,WDTW_Original_g0.1,complete,2.00,0.02,0.78,3466.41,0.29,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,18.00,206.00,87.00,103.67,10.00,Original
1163,DBSCAN,Original,WDTW_Original_g0.1,WDTW_Original_g0.1,eps=9.72963; min_samples=10,NaN,0.01,0.78,3238.14,0.28,2,803,115,688,0.59,"{0: 688, 1: 115}",0.99,NaN,9.73,10.00,12.00,247.00,56.00,105.00,11.00,Original
450,Hierarchical,Original,CDTW_Original_R3,CDTW_Original_R3,average,2.00,0.01,0.78,3407.91,0.29,2,811,124,687,0.62,"{0: 687, 1: 124}",0.00,average,NaN,NaN,23.00,215.00,86.00,108.00,12.00,Original



TOP 10 - DATA MINMAX


,Algoritma,Representasi,Varian,Matriks,Parameter,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise,Linkage,Epsilon,Min_Samples,Rank_SI,Rank_CHI,Rank_DBI,Skor_Ranking,Rank_Akhir,Kelompok_Data
315,KMedoids_PAM,MinMax,WDTW_MinMax_g0.1,WDTW_MinMax_g0.1,-,2.00,0.07,0.35,621.62,1.08,2,811,288,523,0.94,"{0: 523, 1: 288}",0.00,NaN,NaN,NaN,658.00,535.00,496.00,563.00,548.00,MinMax
207,KMedoids_PAM,MinMax,CDTW_MinMax_R5,CDTW_MinMax_R5,-,2.00,0.08,0.35,606.42,1.11,2,811,291,520,0.94,"{0: 520, 1: 291}",0.00,NaN,NaN,NaN,655.00,536.00,503.00,564.67,551.00,MinMax
198,KMedoids_PAM,MinMax,CDTW_MinMax_R3,CDTW_MinMax_R3,-,2.00,0.07,0.35,598.56,1.11,2,811,285,526,0.94,"{0: 526, 1: 285}",0.00,NaN,NaN,NaN,656.00,538.00,506.00,566.67,552.00,MinMax
63,KMeans_DBA,MinMax,CDTW_R3,CDTW_MinMax_R3,-,2.00,41.96,0.35,591.27,1.11,2,811,279,532,0.93,"{0: 532, 1: 279}",0.00,NaN,NaN,NaN,657.00,541.00,504.00,567.33,553.00,MinMax
72,KMeans_DBA,MinMax,CDTW_R5,CDTW_MinMax_R5,-,2.00,36.89,0.34,602.62,1.12,2,811,302,509,0.95,"{0: 509, 1: 302}",0.00,NaN,NaN,NaN,663.00,537.00,508.00,569.33,556.00,MinMax
306,KMedoids_PAM,MinMax,WDTW_MinMax_g0.05,WDTW_MinMax_g0.05,-,2.00,0.08,0.34,591.59,1.11,2,811,286,525,0.94,"{0: 525, 1: 286}",0.00,NaN,NaN,NaN,669.00,540.00,505.00,571.33,558.00,MinMax
324,KMedoids_PAM,MinMax,WDTW_MinMax_g0.5,WDTW_MinMax_g0.5,-,2.00,0.06,0.33,549.54,1.08,2,811,249,562,0.89,"{0: 562, 1: 249}",0.00,NaN,NaN,NaN,681.00,547.00,495.00,574.33,559.00,MinMax
54,KMeans_DBA,MinMax,CDTW_R2,CDTW_MinMax_R2,-,2.00,38.94,0.34,566.50,1.12,2,811,273,538,0.92,"{0: 538, 1: 273}",0.00,NaN,NaN,NaN,668.00,545.00,511.00,574.67,560.00,MinMax
189,KMedoids_PAM,MinMax,CDTW_MinMax_R2,CDTW_MinMax_R2,-,2.00,0.06,0.33,575.77,1.14,2,811,295,516,0.95,"{0: 516, 1: 295}",0.00,NaN,NaN,NaN,674.00,543.00,519.00,578.67,564.00,MinMax
513,Hierarchical,MinMax,CDTW_MinMax_R3,CDTW_MinMax_R3,complete,2.00,0.01,0.33,521.80,1.12,2,811,245,566,0.88,"{0: 566, 1: 245}",0.00,complete,NaN,NaN,677.00,551.00,509.00,579.00,565.00,MinMax



TOP 10 - DATA SCALED


,Algoritma,Representasi,Varian,Matriks,Parameter,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise,Linkage,Epsilon,Min_Samples,Rank_SI,Rank_CHI,Rank_DBI,Skor_Ranking,Rank_Akhir,Kelompok_Data
1079,DBSCAN,Scaled,CDTW_Scaled_R5,CDTW_Scaled_R5,eps=2.94934; min_samples=15,NaN,0.01,0.78,3224.27,0.27,2,803,115,688,0.59,"{0: 688, 1: 115}",0.99,NaN,2.95,15.00,13.00,250.00,34.00,99.00,8.00,Scaled
585,Hierarchical,Scaled,CDTW_Scaled_R5,CDTW_Scaled_R5,complete,2.00,0.02,0.78,3445.32,0.28,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,21.00,210.00,78.00,103.00,9.00,Scaled
423,Hierarchical,Scaled,DTW_Scaled,DTW_Scaled,complete,2.00,0.01,0.77,3401.76,0.28,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,64.00,217.00,53.00,111.33,14.00,Scaled
801,Hierarchical,Scaled,WDTW_Scaled_g0.05,WDTW_Scaled_g0.05,complete,2.00,0.01,0.78,3413.59,0.28,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,51.00,213.00,76.00,113.33,17.00,Scaled
792,Hierarchical,Scaled,WDTW_Scaled_g0.05,WDTW_Scaled_g0.05,average,2.00,0.01,0.77,3367.54,0.28,2,811,124,687,0.62,"{0: 687, 1: 124}",0.00,average,NaN,NaN,60.00,226.00,73.00,119.67,23.00,Scaled
1061,DBSCAN,Scaled,CDTW_Scaled_R3,CDTW_Scaled_R3,eps=3.12515; min_samples=15,NaN,0.02,0.78,3167.19,0.28,2,803,115,688,0.59,"{0: 688, 1: 115}",0.99,NaN,3.13,15.00,28.00,262.00,72.00,120.67,24.00,Scaled
1313,DBSCAN,Scaled,WDTW_Scaled_g0.1,WDTW_Scaled_g0.1,eps=0.826648; min_samples=15,NaN,0.02,0.78,3189.96,0.28,2,804,116,688,0.60,"{0: 688, 1: 116}",0.86,NaN,0.83,15.00,38.00,258.00,68.00,121.33,25.00,Scaled
810,Hierarchical,Scaled,WDTW_Scaled_g0.1,WDTW_Scaled_g0.1,average,2.00,0.01,0.78,3377.60,0.29,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,average,NaN,NaN,46.50,223.50,97.50,122.50,27.00,Scaled
819,Hierarchical,Scaled,WDTW_Scaled_g0.1,WDTW_Scaled_g0.1,complete,2.00,0.01,0.78,3377.60,0.29,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,46.50,223.50,97.50,122.50,27.00,Scaled
1055,DBSCAN,Scaled,CDTW_Scaled_R3,CDTW_Scaled_R3,eps=3.0429; min_samples=10,NaN,0.01,0.78,3136.65,0.28,2,802,114,688,0.59,"{0: 688, 1: 114}",1.11,NaN,3.04,10.00,33.00,271.00,71.00,125.00,35.00,Scaled


## CELL 15. PERBANDINGAN SPESIFIK PER ALGORITMA

In [17]:
for algo in ["KMeans_DBA", "KMedoids_PAM", "Hierarchical", "DBSCAN"]:
    print(f"\nTOP 10 - {algo}")
    display(
        valid[valid["Algoritma"] == algo]
        .sort_values("Rank_Akhir")
        .head(10)
    )



TOP 10 - KMeans_DBA


,Algoritma,Representasi,Varian,Matriks,Parameter,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise,Linkage,Epsilon,Min_Samples,Rank_SI,Rank_CHI,Rank_DBI,Skor_Ranking,Rank_Akhir,Kelompok_Data
0,KMeans_DBA,Original,DTW,DTW_Original,-,2.00,50.85,0.78,3570.66,0.38,2,811,162,649,0.72,"{0: 162, 1: 649}",0.00,NaN,NaN,NaN,14.00,188.00,154.00,118.67,22.00,Original
1,KMeans_DBA,Original,DTW,DTW_Original,-,3.00,41.94,0.69,6556.22,0.42,3,811,124,486,0.86,"{0: 201, 1: 124, 2: 486}",0.00,NaN,NaN,NaN,169.00,14.00,201.00,128.00,38.00,Original
90,KMeans_DBA,Scaled,DTW,DTW_Scaled,-,2.00,36.45,0.78,3543.12,0.39,2,811,165,646,0.73,"{0: 646, 1: 165}",0.00,NaN,NaN,NaN,31.50,192.50,163.50,129.17,42.00,Scaled
27,KMeans_DBA,Original,CDTW_R5,CDTW_Original_R5,-,2.00,48.11,0.77,3607.00,0.37,2,811,153,658,0.70,"{0: 153, 1: 658}",0.00,NaN,NaN,NaN,80.00,177.00,135.00,130.67,45.00,Original
91,KMeans_DBA,Scaled,DTW,DTW_Scaled,-,3.00,51.03,0.68,6326.87,0.43,3,811,126,485,0.86,"{0: 200, 1: 126, 2: 485}",0.00,NaN,NaN,NaN,174.00,19.00,211.00,134.67,52.00,Scaled
117,KMeans_DBA,Scaled,CDTW_R5,CDTW_Scaled_R5,-,2.00,31.89,0.77,3573.30,0.37,2,811,153,658,0.70,"{0: 153, 1: 658}",0.00,NaN,NaN,NaN,94.00,186.00,136.00,138.67,61.00,Scaled
28,KMeans_DBA,Original,CDTW_R5,CDTW_Original_R5,-,3.00,43.07,0.69,6271.05,0.44,3,811,125,483,0.86,"{0: 203, 1: 125, 2: 483}",0.00,NaN,NaN,NaN,167.00,20.00,230.00,139.00,62.00,Original
18,KMeans_DBA,Original,CDTW_R3,CDTW_Original_R3,-,2.00,35.85,0.77,3544.37,0.36,2,811,150,661,0.69,"{0: 661, 1: 150}",0.00,NaN,NaN,NaN,103.00,191.00,133.00,142.33,67.00,Original
108,KMeans_DBA,Scaled,CDTW_R3,CDTW_Scaled_R3,-,2.00,25.52,0.77,3497.34,0.36,2,811,148,663,0.69,"{0: 148, 1: 663}",0.00,NaN,NaN,NaN,110.00,200.00,129.00,146.33,75.00,Scaled
118,KMeans_DBA,Scaled,CDTW_R5,CDTW_Scaled_R5,-,3.00,44.98,0.68,5998.94,0.45,3,811,125,478,0.86,"{0: 478, 1: 125, 2: 208}",0.00,NaN,NaN,NaN,173.00,35.00,238.00,148.67,77.00,Scaled



TOP 10 - KMedoids_PAM


,Algoritma,Representasi,Varian,Matriks,Parameter,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise,Linkage,Epsilon,Min_Samples,Rank_SI,Rank_CHI,Rank_DBI,Skor_Ranking,Rank_Akhir,Kelompok_Data
135,KMedoids_PAM,Original,DTW_Original,DTW_Original,-,2.00,0.08,0.78,3596.27,0.37,2,811,158,653,0.71,"{0: 653, 1: 158}",0.00,NaN,NaN,NaN,11.00,181.00,144.00,112.00,15.00,Original
153,KMedoids_PAM,Scaled,DTW_Scaled,DTW_Scaled,-,2.00,0.05,0.78,3543.12,0.39,2,811,165,646,0.73,"{0: 646, 1: 165}",0.00,NaN,NaN,NaN,31.50,192.50,163.50,129.17,42.00,Scaled
180,KMedoids_PAM,Original,CDTW_Original_R5,CDTW_Original_R5,-,2.00,0.08,0.77,3601.16,0.37,2,811,154,657,0.70,"{0: 657, 1: 154}",0.00,NaN,NaN,NaN,84.00,180.00,138.00,134.00,49.00,Original
270,KMedoids_PAM,Original,WDTW_Original_g0.05,WDTW_Original_g0.05,-,2.00,0.07,0.77,3593.26,0.39,2,811,162,649,0.72,"{0: 649, 1: 162}",0.00,NaN,NaN,NaN,65.00,183.00,159.00,135.67,55.00,Original
136,KMedoids_PAM,Original,DTW_Original,DTW_Original,-,3.00,0.10,0.68,6251.63,0.43,3,811,126,471,0.87,"{0: 214, 1: 126, 2: 471}",0.00,NaN,NaN,NaN,178.00,21.00,215.00,138.00,58.00,Original
181,KMedoids_PAM,Original,CDTW_Original_R5,CDTW_Original_R5,-,3.00,0.12,0.68,6142.81,0.45,3,811,125,475,0.87,"{0: 211, 1: 125, 2: 475}",0.00,NaN,NaN,NaN,172.00,25.00,233.00,143.33,69.00,Original
234,KMedoids_PAM,Scaled,CDTW_Scaled_R5,CDTW_Scaled_R5,-,2.00,0.05,0.77,3562.28,0.37,2,811,155,656,0.70,"{0: 656, 1: 155}",0.00,NaN,NaN,NaN,97.00,190.00,145.00,144.00,70.00,Scaled
342,KMedoids_PAM,Scaled,WDTW_Scaled_g0.05,WDTW_Scaled_g0.05,-,2.00,0.05,0.77,3568.24,0.39,2,811,163,648,0.72,"{0: 648, 1: 163}",0.00,NaN,NaN,NaN,83.00,189.00,162.00,144.67,72.00,Scaled
154,KMedoids_PAM,Scaled,DTW_Scaled,DTW_Scaled,-,3.00,0.13,0.67,6041.58,0.44,3,811,126,467,0.87,"{0: 218, 1: 126, 2: 467}",0.00,NaN,NaN,NaN,187.00,32.00,227.00,148.67,77.00,Scaled
271,KMedoids_PAM,Original,WDTW_Original_g0.05,WDTW_Original_g0.05,-,3.00,0.14,0.68,5961.75,0.45,3,811,126,469,0.87,"{0: 216, 1: 126, 2: 469}",0.00,NaN,NaN,NaN,175.00,37.00,236.00,149.33,80.00,Original



TOP 10 - Hierarchical


,Algoritma,Representasi,Varian,Matriks,Parameter,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise,Linkage,Epsilon,Min_Samples,Rank_SI,Rank_CHI,Rank_DBI,Skor_Ranking,Rank_Akhir,Kelompok_Data
468,Hierarchical,Original,CDTW_Original_R5,CDTW_Original_R5,average,2.00,0.01,0.78,3469.48,0.28,2,811,124,687,0.62,"{0: 687, 1: 124}",0.00,average,NaN,NaN,7.00,205.00,62.00,91.33,1.00,Original
648,Hierarchical,Original,WDTW_Original_g0.05,WDTW_Original_g0.05,average,2.00,0.01,0.78,3488.22,0.28,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,average,NaN,NaN,22.00,202.00,65.00,96.33,4.00,Original
387,Hierarchical,Original,DTW_Original,DTW_Original,complete,2.00,0.01,0.78,3451.77,0.27,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,36.00,209.00,47.00,97.33,6.00,Original
585,Hierarchical,Scaled,CDTW_Scaled_R5,CDTW_Scaled_R5,complete,2.00,0.02,0.78,3445.32,0.28,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,21.00,210.00,78.00,103.00,9.00,Scaled
675,Hierarchical,Original,WDTW_Original_g0.1,WDTW_Original_g0.1,complete,2.00,0.02,0.78,3466.41,0.29,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,18.00,206.00,87.00,103.67,10.00,Original
450,Hierarchical,Original,CDTW_Original_R3,CDTW_Original_R3,average,2.00,0.01,0.78,3407.91,0.29,2,811,124,687,0.62,"{0: 687, 1: 124}",0.00,average,NaN,NaN,23.00,215.00,86.00,108.00,12.00,Original
423,Hierarchical,Scaled,DTW_Scaled,DTW_Scaled,complete,2.00,0.01,0.77,3401.76,0.28,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,64.00,217.00,53.00,111.33,14.00,Scaled
801,Hierarchical,Scaled,WDTW_Scaled_g0.05,WDTW_Scaled_g0.05,complete,2.00,0.01,0.78,3413.59,0.28,2,811,125,686,0.62,"{0: 686, 1: 125}",0.00,complete,NaN,NaN,51.00,213.00,76.00,113.33,17.00,Scaled
477,Hierarchical,Original,CDTW_Original_R5,CDTW_Original_R5,complete,2.00,0.01,0.78,3200.52,0.28,2,811,120,691,0.60,"{0: 691, 1: 120}",0.00,complete,NaN,NaN,44.00,255.00,54.00,117.67,20.00,Original
792,Hierarchical,Scaled,WDTW_Scaled_g0.05,WDTW_Scaled_g0.05,average,2.00,0.01,0.77,3367.54,0.28,2,811,124,687,0.62,"{0: 687, 1: 124}",0.00,average,NaN,NaN,60.00,226.00,73.00,119.67,23.00,Scaled



TOP 10 - DBSCAN


,Algoritma,Representasi,Varian,Matriks,Parameter,K,Running_Time,Silhouette,CHI_MDS,DBI_MDS,Jumlah_Cluster,Jumlah_Dievaluasi,Cluster_Terkecil,Cluster_Terbesar,Balance_Score,Distribusi,Persentase_Noise,Linkage,Epsilon,Min_Samples,Rank_SI,Rank_CHI,Rank_DBI,Skor_Ranking,Rank_Akhir,Kelompok_Data
971,DBSCAN,Original,CDTW_Original_R5,CDTW_Original_R5,eps=35.4894; min_samples=15,NaN,0.01,0.78,3246.21,0.27,2,805,116,689,0.59,"{0: 689, 1: 116}",0.74,NaN,35.49,15.00,9.00,246.00,28.00,94.33,2.00,Original
953,DBSCAN,Original,CDTW_Original_R3,CDTW_Original_R3,eps=37.0134; min_samples=15,NaN,0.02,0.78,3329.58,0.28,2,803,116,687,0.60,"{0: 687, 1: 116}",0.99,NaN,37.01,15.00,3.00,233.00,52.00,96.00,3.00,Original
965,DBSCAN,Original,CDTW_Original_R5,CDTW_Original_R5,eps=34.4093; min_samples=10,NaN,0.02,0.78,3216.15,0.27,2,804,115,689,0.59,"{0: 689, 1: 115}",0.86,NaN,34.41,10.00,10.00,253.00,27.00,96.67,5.00,Original
947,DBSCAN,Original,CDTW_Original_R3,CDTW_Original_R3,eps=36.318; min_samples=10,NaN,0.01,0.78,3309.58,0.28,2,802,115,687,0.59,"{0: 687, 1: 115}",1.11,NaN,36.32,10.00,1.00,240.00,51.00,97.33,6.00,Original
1079,DBSCAN,Scaled,CDTW_Scaled_R5,CDTW_Scaled_R5,eps=2.94934; min_samples=15,NaN,0.01,0.78,3224.27,0.27,2,803,115,688,0.59,"{0: 688, 1: 115}",0.99,NaN,2.95,15.00,13.00,250.00,34.00,99.00,8.00,Scaled
1163,DBSCAN,Original,WDTW_Original_g0.1,WDTW_Original_g0.1,eps=9.72963; min_samples=10,NaN,0.01,0.78,3238.14,0.28,2,803,115,688,0.59,"{0: 688, 1: 115}",0.99,NaN,9.73,10.00,12.00,247.00,56.00,105.00,11.00,Original
959,DBSCAN,Original,CDTW_Original_R5,CDTW_Original_R5,eps=33.2716; min_samples=5,NaN,0.01,0.78,3024.07,0.27,2,794,106,688,0.57,"{0: 688, 1: 106}",2.10,NaN,33.27,5.00,5.00,304.00,23.00,110.67,13.00,Original
970,DBSCAN,Original,CDTW_Original_R5,CDTW_Original_R5,eps=33.121; min_samples=15,NaN,0.01,0.78,2937.81,0.27,2,788,101,687,0.55,"{0: 687, 1: 101}",2.84,NaN,33.12,15.00,2.00,320.00,14.00,112.00,15.00,Original
1150,DBSCAN,Original,WDTW_Original_g0.05,WDTW_Original_g0.05,eps=15.5798; min_samples=15,NaN,0.01,0.78,2942.55,0.26,2,789,102,687,0.56,"{0: 687, 1: 102}",2.71,NaN,15.58,15.00,16.00,318.00,12.00,115.33,18.00,Original
952,DBSCAN,Original,CDTW_Original_R3,CDTW_Original_R3,eps=34.8281; min_samples=15,NaN,0.01,0.78,3050.01,0.27,2,793,106,687,0.57,"{0: 687, 1: 106}",2.22,NaN,34.83,15.00,4.00,298.00,46.00,116.00,19.00,Original


## CELL 16. STABILITY: REPEATED INITIALIZATIONS

In [18]:
# ARI antar-run hanya bermakna untuk algoritma yang memiliki unsur
# random initialization. Di sini diuji untuk KMeans DBA dan KMedoids.
# Untuk Hierarchical dan DBSCAN, hasil pada data/parameter tetap
# deterministik sehingga stability seed tidak relevan.

stability_rows = []
SEEDS = [11, 22, 33, 44, 55]
K_STABILITY = 3

# KMeans DBA stability: DTW standard dan CDTW/DDTW tidak disertakan
# otomatis agar runtime tidak terlalu berat; ubah daftar jika perlu.
for rep, X in X_by_repr.items():
    for variant in ["DTW", "CDTW_R2", "DDTW"]:
        if variant == "DTW":
            X_input = X[:, :, None]
            params = {}
            dist_name = f"DTW_{rep}"
        elif variant == "CDTW_R2":
            X_input = X[:, :, None]
            params = {
                "global_constraint": "sakoe_chiba",
                "sakoe_chiba_radius": 2
            }
            dist_name = f"CDTW_{rep}_R2"
        else:
            X_input = X_derivative[rep][:, :, None]
            params = {}
            dist_name = f"DDTW_{rep}"

        labels_runs = []
        for seed in SEEDS:
            model = TimeSeriesKMeans(
                n_clusters=K_STABILITY,
                metric="dtw",
                metric_params=params,
                max_iter=10,
                max_iter_barycenter=10,
                n_init=1,
                random_state=seed
            )
            labels_runs.append(model.fit_predict(X_input))

        ari_values = [
            adjusted_rand_score(labels_runs[i], labels_runs[j])
            for i in range(len(labels_runs))
            for j in range(i + 1, len(labels_runs))
        ]
        stability_rows.append({
            "Algoritma": "KMeans_DBA",
            "Matriks": dist_name,
            "K": K_STABILITY,
            "ARI_Rata2": np.mean(ari_values),
            "ARI_SD": np.std(ari_values)
        })

# KMedoids stability
for name, D in distance_matrices.items():
    labels_runs = []
    for seed in SEEDS:
        model = KMedoids(
            n_clusters=K_STABILITY,
            metric="precomputed",
            method="pam",
            init="k-medoids++",
            random_state=seed
        )
        labels_runs.append(model.fit_predict(D))

    ari_values = [
        adjusted_rand_score(labels_runs[i], labels_runs[j])
        for i in range(len(labels_runs))
        for j in range(i + 1, len(labels_runs))
    ]
    stability_rows.append({
        "Algoritma": "KMedoids_PAM",
        "Matriks": name,
        "K": K_STABILITY,
        "ARI_Rata2": np.mean(ari_values),
        "ARI_SD": np.std(ari_values)
    })

tabel_stability = pd.DataFrame(stability_rows)
tabel_stability.to_csv(
    os.path.join(OUTPUT_DIR, "stability_ARI.csv"), index=False
)
display(tabel_stability.sort_values("ARI_Rata2", ascending=False))


,Algoritma,Matriks,K,ARI_Rata2,ARI_SD
17,KMedoids_PAM,CDTW_MinMax_R5,3,1.00,0.00
18,KMedoids_PAM,CDTW_Scaled_R2,3,1.00,0.00
9,KMedoids_PAM,DTW_Original,3,1.00,0.00
11,KMedoids_PAM,DTW_Scaled,3,1.00,0.00
33,KMedoids_PAM,WDTW_Scaled_g0.1,3,1.00,0.00
32,KMedoids_PAM,WDTW_Scaled_g0.05,3,1.00,0.00
35,KMedoids_PAM,WDTW_Scaled_g1.0,3,1.00,0.00
26,KMedoids_PAM,WDTW_Original_g0.5,3,1.00,0.00
15,KMedoids_PAM,CDTW_MinMax_R2,3,1.00,0.00
14,KMedoids_PAM,CDTW_Original_R5,3,1.00,0.00


## CELL 17. FINAL TOP-3 SUMMARY

In [19]:
top3 = valid.sort_values("Rank_Akhir").head(3).copy()
kolom_top3 = [
    "Algoritma", "Kelompok_Data", "Matriks", "Parameter", "K",
    "Jumlah_Cluster", "Silhouette", "CHI_MDS", "DBI_MDS",
    "Persentase_Noise", "Balance_Score", "Running_Time", "Rank_Akhir"
]
kolom_top3 = [c for c in kolom_top3 if c in top3.columns]
display(top3[kolom_top3].round(4))

top3[kolom_top3].to_csv(
    os.path.join(OUTPUT_DIR, "TOP_3_KOMBINASI.csv"), index=False
)


,Algoritma,Kelompok_Data,Matriks,Parameter,K,Jumlah_Cluster,Silhouette,CHI_MDS,DBI_MDS,Persentase_Noise,Balance_Score,Running_Time,Rank_Akhir
468,Hierarchical,Original,CDTW_Original_R5,average,2.00,2,0.78,3469.48,0.28,0.00,0.62,0.01,1.00
971,DBSCAN,Original,CDTW_Original_R5,eps=35.4894; min_samples=15,NaN,2,0.78,3246.21,0.27,0.74,0.59,0.01,2.00
953,DBSCAN,Original,CDTW_Original_R3,eps=37.0134; min_samples=15,NaN,2,0.78,3329.58,0.28,0.99,0.60,0.02,3.00


## CELL 18. EXPORT LABEL CLUSTER TOP MODEL

In [20]:
# Label model terbaik disimpan dengan Product_Code.
best = valid.sort_values("Rank_Akhir").iloc[0]
best_algo = best["Algoritma"]
best_matrix = best["Matriks"]

if best_algo == "KMeans_DBA":
    key = (best_matrix, int(best["K"]))
    best_labels = kmeans_labels[key]
elif best_algo == "KMedoids_PAM":
    key = (best_matrix, int(best["K"]))
    best_labels = kmedoids_labels[key]
elif best_algo == "Hierarchical":
    key = (best_matrix, best["Linkage"], int(best["K"]))
    best_labels = hier_labels[key]
else:
    key = (best_matrix, float(best["Epsilon"]), int(best["Min_Samples"]))
    best_labels = dbscan_labels[key]

hasil_produk_terbaik = pd.DataFrame({
    "Product_Code": product_code,
    "Cluster": best_labels
})
hasil_produk_terbaik.to_csv(
    os.path.join(OUTPUT_DIR, "label_cluster_model_terbaik.csv"),
    index=False
)

print("Selesai. Seluruh output tersimpan di:", OUTPUT_DIR)


Selesai. Seluruh output tersimpan di: hasil_clustering_dtw


## CELL 19. EXPORT ASSET UNTUK STREAMLIT

Cell ini mengekspor matriks jarak, ranking evaluasi, dan data ringkasan yang dibutuhkan aplikasi Streamlit. Jalankan setelah seluruh eksperimen pada CELL 1–18 selesai.

**Catatan:** `all_distance_matrices.npz` berukuran cukup besar. Pastikan file ini ikut disimpan di repository (atau gunakan Git LFS bila ukuran melewati batas GitHub). Aplikasi akan membaca ulang matriks ini, bukan menghitung ulang seluruh 27 matriks setiap kali dibuka.

In [26]:
from pathlib import Path
import shutil

# Folder asset yang akan disalin ke repository Streamlit
STREAMLIT_DIR = Path('streamlit_assets')
STREAMLIT_DIR.mkdir(exist_ok=True)

# Simpan matriks jarak lengkap (DTW, CDTW, DDTW, WDTW)
matrix_file = Path(OUTPUT_DIR) / 'all_distance_matrices.npz'
if not matrix_file.exists():
    np.savez_compressed(matrix_file, **distance_matrices, product_code=product_code)
shutil.copy2(matrix_file, STREAMLIT_DIR / 'all_distance_matrices.npz')

# Ekspor tabel hasil eksperimen dan ranking
tabel_semua.to_csv(STREAMLIT_DIR / 'evaluasi_seluruh_model.csv', index=False)
valid.sort_values('Rank_Akhir').to_csv(STREAMLIT_DIR / 'ranking_model_valid.csv', index=False)

# Simpan dataset yang telah digunakan dalam eksperimen
df.to_excel(STREAMLIT_DIR / 'Sales transaction data.xlsx', index=False)

print('Asset Streamlit tersimpan di:', STREAMLIT_DIR.resolve())
print('File:', [p.name for p in STREAMLIT_DIR.iterdir()])


Asset Streamlit tersimpan di: /content/streamlit_assets
File: ['ranking_model_valid.csv', 'Sales transaction data.xlsx', 'all_distance_matrices.npz', 'evaluasi_seluruh_model.csv']


## CELL 20. SIAPKAN FILE APLIKASI STREAMLIT

Simpan `app.py` dan `requirements.txt` yang disediakan bersama notebook ini ke root repository GitHub. Letakkan `Sales transaction data.xlsx` dan `all_distance_matrices.npz` pada struktur folder yang sesuai. Contoh struktur repository:

```text
sales-clustering-dashboard/
├── app.py
├── requirements.txt
├── Sales transaction data.xlsx
└── hasil_clustering_dtw/
    └── all_distance_matrices.npz
```

Jalankan lokal dengan `streamlit run app.py`. Untuk Streamlit Community Cloud, hubungkan repository GitHub, pilih branch, lalu set Main file path menjadi `app.py`.

In [27]:
from google.colab import files

files.download(
    'streamlit_assets/all_distance_matrices.npz'
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>